In [0]:
%pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 86.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [statsmodels]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ==============================================================================
# CAMADA SILVER E GOLD: ETL, NLP (CHAVE JOIN) E MODELAGEM ESTATÍSTICA
# ==============================================================================
import os
import pandas as pd
import unicodedata
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf

CAMINHO_VOLUME = "/Volumes/workspace/default/fontes_tcc"

print("="*75)
print(" 1. CARREGANDO AS BASES BRUTAS (IBGE E S2iD)")
print("="*75)

# 1.1 Lendo IBGE API
df_pop = pd.read_csv(os.path.join(CAMINHO_VOLUME, "IBGE_SIDRA_Tabela4714_Populacao.csv"), sep=";")
df_esg = pd.read_csv(os.path.join(CAMINHO_VOLUME, "IBGE_SIDRA_Tabela9397_Esgoto.csv"), sep=";")
df_agu = pd.read_csv(os.path.join(CAMINHO_VOLUME, "IBGE_SIDRA_Tabela6909_Agua.csv"), sep=";")

# 1.2 Lendo S2iD (Utilizando os arquivos consolidados que estão na raiz do Volume!)
df_danos = pd.read_csv(os.path.join(CAMINHO_VOLUME, "S2iD_Danos_Informados_Bruto_2021_2025.csv"), sep=";")
df_rec = pd.read_csv(os.path.join(CAMINHO_VOLUME, "S2iD_Reconhecimentos_Bruto_2021_2025.csv"), sep=";")

print("[SUCESSO] Bases carregadas.")

print("\n" + "="*75)
print(" 2. PROCESSAMENTO DE LINGUAGEM NATURAL (NLP) E PADRONIZAÇÃO")
print("="*75)

# Função NLP para remover acentos e padronizar
def limpar_nome(nome):
    if not isinstance(nome, str): return nome
    nome = ''.join(c for c in unicodedata.normalize('NFD', nome) if unicodedata.category(c) != 'Mn')
    return nome.strip().upper()

# 2.1 Aplicando NLP no IBGE
for df_ibge, nome_valor in zip([df_pop, df_esg, df_agu], ['Populacao', 'Esgoto_Total', 'Agua_Total']):
    df_ibge.rename(columns={'Valor': nome_valor}, inplace=True)
    df_ibge['Cidade_Limpa'] = df_ibge['Município'].apply(lambda x: limpar_nome(x.rsplit(' - ', 1)[0]))
    df_ibge['UF_Limpa'] = df_ibge['Município'].apply(lambda x: x.rsplit(' - ', 1)[1].strip().upper())
    df_ibge['Chave_Join'] = df_ibge['Cidade_Limpa'] + "_" + df_ibge['UF_Limpa']

df_pop_limpo = df_pop[['Chave_Join', 'Município (Código)', 'Populacao']].copy()
df_esg_limpo = df_esg[['Chave_Join', 'Esgoto_Total']].copy()
df_agu_limpo = df_agu[['Chave_Join', 'Agua_Total']].copy()

# 2.2 Aplicando NLP no S2iD
df_danos['Chave_Join'] = df_danos['Município'].apply(limpar_nome) + "_" + df_danos['UF'].astype(str).str.strip().str.upper()
df_rec['Chave_Join'] = df_rec['Município'].apply(limpar_nome) + "_" + df_rec['UF'].astype(str).str.strip().str.upper()

# 2.3 Agrupando Desalojados e Contando Decretos
# Força a coluna DH_Desalojados a ser número antes de somar
df_danos['DH_Desalojados'] = pd.to_numeric(df_danos['DH_Desalojados'], errors='coerce').fillna(0)
df_danos_agrupado = df_danos.groupby('Chave_Join')['DH_Desalojados'].sum().reset_index()
df_rec_agrupado = df_rec.groupby('Chave_Join').size().reset_index(name='Qtd_Decretos')

print("\n" + "="*75)
print(" 3. CRUZAMENTO RELACIONAL (JOIN) E ENGENHARIA DE ATRIBUTOS")
print("="*75)

# Empilhando as bases
df_gold = pd.merge(df_pop_limpo, df_esg_limpo, on='Chave_Join', how='left')
df_gold = pd.merge(df_gold, df_agu_limpo, on='Chave_Join', how='left')
df_gold = pd.merge(df_gold, df_danos_agrupado, on='Chave_Join', how='left')
df_gold = pd.merge(df_gold, df_rec_agrupado, on='Chave_Join', how='left')

# CORREÇÃO DO ERRO (TypeError): Forçando todas as colunas do IBGE a serem números reais
df_gold['Populacao'] = pd.to_numeric(df_gold['Populacao'], errors='coerce')
df_gold['Esgoto_Total'] = pd.to_numeric(df_gold['Esgoto_Total'], errors='coerce')
df_gold['Agua_Total'] = pd.to_numeric(df_gold['Agua_Total'], errors='coerce')

# Tratamento de Nulos
df_gold['DH_Desalojados'] = df_gold['DH_Desalojados'].fillna(0).astype(int)
df_gold['Qtd_Decretos'] = df_gold['Qtd_Decretos'].fillna(0).astype(int)
# Mantém apenas cidades válidas (População > 0 e que tenham dados de saneamento)
df_gold = df_gold[df_gold['Populacao'] > 0].dropna(subset=['Esgoto_Total', 'Agua_Total']).copy()

# Criando as Taxas Relativas e a Variável de Controle (Exposição)
df_gold['Taxa_Desalojados'] = (df_gold['DH_Desalojados'] / df_gold['Populacao']) * 100000
df_gold['Perc_Esgoto'] = (df_gold['Esgoto_Total'] / df_gold['Populacao']) * 100
df_gold['Perc_Agua'] = (df_gold['Agua_Total'] / df_gold['Populacao']) * 100
df_gold['Log_Pop'] = np.log(df_gold['Populacao'])

print(f"[AUDITORIA] Municípios validados na Matriz Analítica: {len(df_gold)}")
print(f"[AUDITORIA] Curtose de Desalojados: {df_gold['DH_Desalojados'].kurtosis():.2f} (Maior que 300 = Sobredispersão extrema!)")

print("\n" + "="*75)
print(" 4. MODELAGEM ESTATÍSTICA AVANÇADA")
print("="*75)

# Limpeza Matemática para o Statsmodels não explodir (RuntimeWarning: divide by zero)
# Filtra apenas registros com população válida e substitui zeros no esgoto por um valor mínimo
df_glm = df_gold.replace([np.inf, -np.inf], np.nan).dropna(subset=['Perc_Esgoto', 'Log_Pop', 'Qtd_Decretos']).copy()
# Adiciona 0.001 para evitar log(0) internamente no statsmodels
df_glm['Perc_Esgoto_GLM'] = df_glm['Perc_Esgoto'] + 0.001 

# 4.1 Correlação de Spearman
corr_esgoto, p_esgoto = stats.spearmanr(df_glm['Perc_Esgoto_GLM'], df_glm['Taxa_Desalojados'])
print(f"-> Correlação Spearman (Saneamento x Taxa Desalojados): ρ = {corr_esgoto:.4f} | p = {p_esgoto:.4e}")

# 4.2 Regressão Binomial Negativa (O Paradoxo da Notificação)
try:
    # Usando o parâmetro bfgs para melhorar a convergência do modelo e evitar o erro do log
    modelo_decretos = smf.negativebinomial('Qtd_Decretos ~ Perc_Esgoto_GLM + Log_Pop', data=df_glm).fit(method='bfgs', disp=False)
    beta = modelo_decretos.params['Perc_Esgoto_GLM']
    p_beta = modelo_decretos.pvalues['Perc_Esgoto_GLM']
    print(f"-> Regressão Binomial Negativa (Qtd_Decretos): Coeficiente β(Saneamento) = {beta:.4f} | p = {p_beta:.4e}")
    print(modelo_decretos.summary())
except Exception as e:
    print(f"Erro na regressão GLM: {e}")

# ==============================================================================
# 5. MATERIALIZANDO A MATRIZ NA CAMADA GOLD (DELTA TABLE PARA POWER BI)
# ==============================================================================
print("\n" + "="*75)
print(" 5. MATERIALIZANDO A MATRIZ (DELTA TABLE)")
print("="*75)

try:
    # 1. Correção do Erro Delta: Renomeando colunas com caracteres inválidos
    df_gold.rename(columns={'Município (Código)': 'Municipio_Codigo'}, inplace=True)

    # 2. Converte o DataFrame do Pandas (df_gold) para o formato Spark (nativo)
    spark_df = spark.createDataFrame(df_gold)

    # 3. Define o nome da tabela no catálogo padrão do Databricks
    nome_tabela = "default.matriz_analitica_tcc"
    
    # 4. Salva a matriz analítica como uma tabela oficial (Delta Table)
    spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome_tabela)

    print(f"[SUCESSO] Tabela '{nome_tabela}' materializada com sucesso no banco de dados!")
    print("O Databricks agora está pronto para conexão direta em nuvem com o Power BI via Endpoint SQL.")
    
except Exception as e:
    print(f"Erro ao salvar a tabela Delta: {e}")

 1. CARREGANDO AS BASES BRUTAS (IBGE E S2iD)
[SUCESSO] Bases carregadas.

 2. PROCESSAMENTO DE LINGUAGEM NATURAL (NLP) E PADRONIZAÇÃO

 3. CRUZAMENTO RELACIONAL (JOIN) E ENGENHARIA DE ATRIBUTOS
[AUDITORIA] Municípios validados na Matriz Analítica: 5538
[AUDITORIA] Curtose de Desalojados: 976.75 (Maior que 300 = Sobredispersão extrema!)

 4. MODELAGEM ESTATÍSTICA AVANÇADA
-> Correlação Spearman (Saneamento x Taxa Desalojados): ρ = 0.0257 | p = 5.5562e-02
-> Regressão Binomial Negativa (Qtd_Decretos): Coeficiente β(Saneamento) = -0.0060 | p = 6.3529e-40
                     NegativeBinomial Regression Results                      
Dep. Variable:           Qtd_Decretos   No. Observations:                 5538
Model:               NegativeBinomial   Df Residuals:                     5535
Method:                           MLE   Df Model:                            2
Date:                Mon, 05 Oct 2026   Pseudo R-squ.:                0.006431
Time:                        02:03:53   Log-Lik

/local_disk0/.ephemeral_nfs/envs/pythonEnv-65273a27-882a-4510-9bd6-c7bbd8e0e3d0/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:3709: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size * np.log(prob) + endog * np.log(1 - prob)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-65273a27-882a-4510-9bd6-c7bbd8e0e3d0/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:3709: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size * np.log(prob) + endog * np.log(1 - prob)


[SUCESSO] Tabela 'default.matriz_analitica_tcc' materializada com sucesso no banco de dados!
O Databricks agora está pronto para conexão direta em nuvem com o Power BI via Endpoint SQL.
